In [ ]:
# Setup
from pathlib import Path
import sys, json

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np

from data import load_data
from model import build_models, split_data, cross_validate_model, evaluate_model, save_model, RANDOM_SEED
from utils import save_json, save_fig, pretty_print_cv, plot_confusion

ARTEFACTS_DIR = PROJECT_ROOT / 'artefacts'
ARTEFACTS_DIR.mkdir(exist_ok=True)

np.random.seed(RANDOM_SEED)
print(f"Random seed: {RANDOM_SEED}")

In [ ]:
# 1. Load data + split
df = load_data()
print(f"Loaded {len(df)} rows")

X_train, X_test, y_train, y_test = split_data(df)
print(f"Train: {len(X_train)} rows; churn rate = {y_train.mean():.2%}")
print(f"Test:  {len(X_test)} rows; churn rate = {y_test.mean():.2%}")

In [ ]:
# 2. Train + cross-validate all three models
models = build_models()
cv_results_all = {}
test_results_all = {}

for name, pipe in models.items():
    print(f"\n=== {name} ===")
    pipe.fit(X_train, y_train)
    cv = cross_validate_model(pipe, X_train, y_train, cv_folds=5)
    print(pretty_print_cv(cv))
    test_res = evaluate_model(pipe, X_test, y_test)
    print(f"  Test F1 = {test_res['f1']:.4f} | ROC-AUC = {test_res['roc_auc']:.4f}")
    cv_results_all[name] = cv
    test_results_all[name] = test_res

In [ ]:
# 3. Pick the best model by test F1
best_name = max(test_results_all, key=lambda k: test_results_all[k]['f1'])
best_pipe = models[best_name]
print(f"Best model: {best_name}")
print(f"  Test F1 = {test_results_all[best_name]['f1']:.4f}")
print(f"  Test ROC-AUC = {test_results_all[best_name]['roc_auc']:.4f}")

# 4. Save the best model for the Streamlit app
saved = save_model(best_pipe, 'best_model', ARTEFACTS_DIR)
print(f"Saved → {saved}")

In [ ]:
# 5. Save the full results bundle for the evaluation notebook
save_json({
    'cv_results': cv_results_all,
    'test_results': {k: {kk: vv for kk, vv in v.items() if kk != 'classification_report'}
                     for k, v in test_results_all.items()},
    'best_model': best_name,
    'feature_columns': X_train.columns.tolist(),
    'random_seed': RANDOM_SEED,
}, ARTEFACTS_DIR / 'model_results.json')
print(f"Saved → {ARTEFACTS_DIR / 'model_results.json'}")